In [1]:
pip install imblearn catboost optuna

Note: you may need to restart the kernel to use updated packages.


In [21]:
# ============================================================
# 🧠 FINAL MODEL: CatBoost + SMOTE-Tomek + Class Weights + Optuna + Threshold Tuning
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
)
from catboost import CatBoostClassifier
from imblearn.combine import SMOTETomek
import optuna
import warnings
warnings.filterwarnings("ignore")

# ===========================================
# 🧹 1. Load & Cleaning
# ===========================================
df = pd.read_csv("data-bank.csv").dropna().drop_duplicates()

X = df.drop(columns=["Bankrupt?"])
y = df["Bankrupt?"]

# ===========================================
# ✂️ 2. Split & Scaling
# ===========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ===========================================
# ⚖️ 3. Balancing (SMOTE + Tomek Links)
# ===========================================
smote_tomek = SMOTETomek(random_state=42)
X_res, y_res = smote_tomek.fit_resample(X_train_scaled, y_train)

print("Jumlah data setelah balancing:")
print(y_res.value_counts())

# ===========================================
# 🎯 4. Hyperparameter Tuning (Optuna)
# ===========================================
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 400, 1200),
        'depth': trial.suggest_int('depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 8),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'border_count': trial.suggest_int('border_count', 32, 128),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
        'verbose': False,
        'random_seed': 42,
        # 💥 class_weights penting banget
        'class_weights': [1, 5]
    }

    model = CatBoostClassifier(**params)
    model.fit(X_res, y_res)
    y_pred = model.predict(X_test_scaled)
    
    # 🎯 Fokus ke F1 kelas minoritas
    return f1_score(y_test, y_pred, pos_label=1)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

best_params = study.best_params
best_params['class_weights'] = [1, 5]  # tetap dipakai di final model
print("\n✅ Best Parameters:", best_params)

# ===========================================
# 🧠 5. Train Final Model
# ===========================================
best_model = CatBoostClassifier(**best_params, verbose=False, random_seed=42)
best_model.fit(X_res, y_res)

# ===========================================
# 🎯 6. Threshold Tuning (based on F1)
# ===========================================
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_opt = (y_proba >= best_threshold).astype(int)

# ===========================================
# 📊 7. Evaluasi Akhir
# ===========================================
print(f"\nOptimal threshold: {best_threshold:.3f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_opt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_opt, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

precision_bankruptcy = precision_score(y_test, y_pred_opt, pos_label=1)
recall_bankruptcy = recall_score(y_test, y_pred_opt, pos_label=1)
f1_bankruptcy = f1_score(y_test, y_pred_opt, pos_label=1)

print(f"\nPrecision (Bankruptcy=1): {precision_bankruptcy:.3f}")
print(f"Recall (Bankruptcy=1): {recall_bankruptcy:.3f}")
print(f"F1-score (Bankruptcy=1): {f1_bankruptcy:.3f}")


[I 2025-11-13 21:45:15,761] A new study created in memory with name: no-name-3320e7cc-2869-403b-bbdb-5e106ff6d247


Jumlah data setelah balancing:
Bankrupt?
0    5278
1    5278
Name: count, dtype: int64


[I 2025-11-13 21:45:30,309] Trial 0 finished with value: 0.4393939393939394 and parameters: {'iterations': 466, 'depth': 10, 'learning_rate': 0.0406748282918263, 'l2_leaf_reg': 3.159556202004244, 'bagging_temperature': 0.22477416714013032, 'border_count': 95, 'random_strength': 0.49112517023870983}. Best is trial 0 with value: 0.4393939393939394.
[I 2025-11-13 21:45:33,918] Trial 1 finished with value: 0.3333333333333333 and parameters: {'iterations': 637, 'depth': 6, 'learning_rate': 0.04109683777216985, 'l2_leaf_reg': 7.1498150489591294, 'bagging_temperature': 0.4460443725142186, 'border_count': 104, 'random_strength': 0.8034022988882958}. Best is trial 0 with value: 0.4393939393939394.
[I 2025-11-13 21:46:09,079] Trial 2 finished with value: 0.42748091603053434 and parameters: {'iterations': 1041, 'depth': 10, 'learning_rate': 0.0660684709143409, 'l2_leaf_reg': 4.424925653851274, 'bagging_temperature': 0.37089233502680363, 'border_count': 100, 'random_strength': 0.8512468890895121}.


✅ Best Parameters: {'iterations': 968, 'depth': 9, 'learning_rate': 0.08624229735799889, 'l2_leaf_reg': 3.3028010390105336, 'bagging_temperature': 0.9952760307198953, 'border_count': 55, 'random_strength': 0.28605491581057463, 'class_weights': [1, 5]}

Optimal threshold: 0.944

Confusion Matrix:
 [[1310   10]
 [  22   22]]

Classification Report:
               precision    recall  f1-score   support

           0      0.983     0.992     0.988      1320
           1      0.688     0.500     0.579        44

    accuracy                          0.977      1364
   macro avg      0.835     0.746     0.783      1364
weighted avg      0.974     0.977     0.975      1364

ROC-AUC: 0.9368801652892562

Precision (Bankruptcy=1): 0.688
Recall (Bankruptcy=1): 0.500
F1-score (Bankruptcy=1): 0.579
